In [ ]:
!pip install -q torch torchaudio librosa pyyaml seaborn scikit-learn tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

repo_url = "https://github.com/Jessitoii/cnn-based-genre-recognition-fma"
repo_dir = "/content/cnn-based-genre-recognition-fma"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    %cd {repo_dir}
    !git pull


In [ ]:
import sys
import os

# Set working directory to the ai folder of the repo
ai_dir = "/content/cnn-based-genre-recognition-fma/ai"
os.chdir(ai_dir)
print("Working directory set to:", os.getcwd())

# Add ai/src to sys.path so we can import from it
src_dir = os.path.join(ai_dir, "src")
if src_dir not in sys.path:
    sys.path.append(src_dir)


In [ ]:
import zipfile, os

cfg = load_config("config.yaml")

# Manually remap paths to Google Drive for Colab preprocessing
drive_data = cfg["colab"]["drive_data"]

# fma_metadata
with zipfile.ZipFile(f"{drive_data}/fma_metadata.zip", 'r') as z:
    z.extractall(f"{drive_data}/raw/")

# fma_small
with zipfile.ZipFile(f"{drive_data}/fma_small.zip", 'r') as z:
    z.extractall(f"{drive_data}/raw/")

print("Done.")

In [ ]:
from tqdm.notebook import tqdm
import dataset
dataset.tqdm = tqdm
from dataset import load_config, preprocess_and_save

cfg["paths"]["data_processed"] = f"{drive_data}/processed/spectrograms"
cfg["paths"]["metadata"]       = f"{drive_data}/raw/fma_metadata/tracks.csv"
cfg["paths"]["data_raw"]       = f"{drive_data}/raw/fma_small"

preprocess_and_save(cfg, out_dir=cfg["paths"]["data_processed"])


In [ ]:
from tqdm.notebook import tqdm
import train as train_mod
train_mod.tqdm = tqdm
from train import train

# Run training and collect history dictionary
history = train(config_path="config.yaml", colab=True)


In [ ]:
from evaluate import evaluate

# Run evaluation
test_acc = evaluate(config_path="config.yaml", colab=True)


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], label='Train Loss')
plt.plot(epochs, history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], label='Train Accuracy')
plt.plot(epochs, history['val_acc'], label='Val Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()
